# Task 1.2 — Bronze Ingestion: Quiz Attempts + Student Enrollments
## Notebook: 02_bronze_quiz_enrollments

**Layer:** Bronze (raw ingestion)

**What this notebook does:**
- Ingests `quiz_attempts.csv` incrementally using max(submit_ts) watermark → `bronze.quiz_attempts_bronze`
- Ingests `student_enrollments.csv` daily delta using MERGE INTO on enrollment_id → `bronze.enrollments_bronze`
- Adds metadata: `_source_file`, `_load_ts`, `_schema_version`, `_last_modified_ts`

**Business Rules enforced:**
- score_obtained must be between 0 and max_score
- status=IN_PROGRESS with non-null submit_ts → logical inconsistency flag
- All course_ids in enrollments must exist in course_catalog_bronze (referential integrity)
- student_id and course_id must not be NULL in either table

**Run order:** Run all cells top-to-bottom. Safe to re-run (idempotent).

In [0]:
# ============================================================
# CELL 2 — Catalog config and all path/table name constants
# Run this cell first every time before executing any other cell
# ============================================================

# --- Catalog & schema ---
CATALOG = "mini_project_grp6"
BRONZE  = "bronze"

# --- Volume source paths ---
QUIZ_RAW_PATH        = f"/Volumes/{CATALOG}/{BRONZE}/quiz_attempts_raw/"
ENROLLMENTS_RAW_PATH = f"/Volumes/{CATALOG}/{BRONZE}/enrollments_raw/"

# --- Target Delta table names (fully qualified) ---
QUIZ_BRONZE_TABLE    = f"{CATALOG}.{BRONZE}.quiz_attempts_bronze"
ENROLL_BRONZE_TABLE  = f"{CATALOG}.{BRONZE}.enrollments_bronze"
CATALOG_BRONZE_TABLE = f"{CATALOG}.{BRONZE}.course_catalog_bronze"
DQ_AUDIT_TABLE       = f"{CATALOG}.audit.dq_audit"

# --- Schema version tag ---
SCHEMA_VERSION = "v1.0"

# --- Set default catalog + schema for this session ---
spark.catalog.setCurrentCatalog(CATALOG)
spark.catalog.setCurrentDatabase(BRONZE)

print(f"✅ Catalog  : {CATALOG}")
print(f"✅ Schema   : {BRONZE}")
print(f"✅ Quiz path       : {QUIZ_RAW_PATH}")
print(f"✅ Enrollments path: {ENROLLMENTS_RAW_PATH}")

In [0]:
# ============================================================
# CELL 3 — All imports needed for this notebook
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType, DoubleType
)
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("✅ Imports successful")

## Part A — Quiz Attempts Bronze (`quiz_attempts_bronze`)

**Source:** `quiz_attempts.csv` from Volume `quiz_attempts_raw/`
**Method:** Incremental batch using max(submit_ts) as high-watermark

- On first run: reads all rows (no prior watermark)
- On subsequent runs: reads only rows where submit_ts > last max(submit_ts) in the table
- Partitioned by derived column `attempt_date` from submit_ts
- Adds: `_source_file`, `_load_ts`, `_schema_version`

In [0]:
# ============================================================
# CELL 5 — Explicit schema for quiz_attempts.csv
# CSV has no self-describing types — always define explicitly
# ============================================================

quiz_schema = StructType([
    StructField("attempt_id",     StringType(),    nullable=False),
    StructField("student_id",     StringType(),    nullable=True),
    StructField("course_id",      StringType(),    nullable=True),
    StructField("quiz_id",        StringType(),    nullable=True),
    StructField("quiz_name",      StringType(),    nullable=True),
    StructField("attempt_number", IntegerType(),   nullable=True),
    StructField("start_ts",       TimestampType(), nullable=True),
    StructField("submit_ts",      TimestampType(), nullable=True),
    StructField("score_obtained", IntegerType(),   nullable=True),
    StructField("max_score",      IntegerType(),   nullable=True),
    StructField("pass_threshold", IntegerType(),   nullable=True),
    StructField("status",         StringType(),    nullable=True),
    StructField("answers_json",   StringType(),    nullable=True),
])

print("✅ Quiz schema defined with", len(quiz_schema.fields), "fields")

In [0]:
# ============================================================
# CELL 6 — Watermark: read max(submit_ts) from bronze table
#
# Logic:
#   - If table does not exist yet → watermark = None (read everything)
#   - If table exists             → watermark = max(submit_ts) already loaded
#
# This ensures we never re-process rows already in the table
# ============================================================

def get_watermark(table_name: str, ts_column: str):
    try:
        wm = (
            spark.table(table_name)
            .agg(F.max(F.col(ts_column)).alias("max_ts"))
            .collect()[0]["max_ts"]
        )
        print(f"✅ Watermark found: {wm}  (will load rows AFTER this timestamp)")
        return wm
    except Exception:
        print("ℹ  Table does not exist yet — first run, loading ALL rows")
        return None

quiz_watermark = get_watermark(QUIZ_BRONZE_TABLE, "submit_ts")

In [0]:
# ============================================================
# CELL 7 — Incremental read of quiz_attempts.csv
#          Apply watermark filter → add metadata → APPEND to Delta
# ============================================================

# Read all CSV files in the volume folder with explicit schema
quiz_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")      # never infer — use explicit schema
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(quiz_schema)
    .load(QUIZ_RAW_PATH)
)

# Apply watermark filter — only keep rows newer than last loaded submit_ts
if quiz_watermark is not None:
    quiz_incremental = quiz_raw.filter(F.col("submit_ts") > F.lit(quiz_watermark))
else:
    quiz_incremental = quiz_raw   # first run — take everything

# Add Bronze metadata + derive partition column
quiz_bronze_df = (
    quiz_incremental
    .withColumn("_source_file",    F.input_file_name())
    .withColumn("_load_ts",        F.current_timestamp())
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn("attempt_date",    F.to_date(F.col("submit_ts")))  # partition key
)

new_rows = quiz_bronze_df.count()
print(f"ℹ  New rows to load (after watermark): {new_rows:,}")

if new_rows > 0:
    (
        quiz_bronze_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .partitionBy("attempt_date")
        .saveAsTable(QUIZ_BRONZE_TABLE)
    )
    print(f"✅ Written to: {QUIZ_BRONZE_TABLE}")
else:
    print("ℹ  No new rows to append — table is already up to date")

In [0]:
# ============================================================
# CELL 8 — Quick verification of quiz_attempts_bronze
# ============================================================

quiz_df = spark.table(QUIZ_BRONZE_TABLE)

print("── quiz_attempts_bronze ──────────────────────────")
print(f"Total rows   : {quiz_df.count():,}")
print(f"Columns      : {len(quiz_df.columns)}")
print()

print("── status distribution ───────────────────────────")
quiz_df.groupBy("status").count().orderBy(F.desc("count")).show(truncate=False)

print("── attempt_number distribution ───────────────────")
quiz_df.groupBy("attempt_number").count().orderBy("attempt_number").show()

print("── metadata sample ───────────────────────────────")
quiz_df.select(
    "attempt_id", "submit_ts", "_load_ts", "_schema_version", "attempt_date"
).show(3, truncate=50)

## Part B — Student Enrollments Bronze (`enrollments_bronze`)

**Source:** `student_enrollments.csv` from Volume `enrollments_raw/`
**Method:** MERGE INTO on `enrollment_id` (upsert — handles both new and updated rows)

- WHEN MATCHED (enrollment_id exists): UPDATE all columns + `_last_modified_ts`
- WHEN NOT MATCHED (new enrollment):   INSERT full row
- This preserves history correctly — a student moving from ACTIVE to DROPPED
  is captured as an UPDATE, not a duplicate row

In [0]:
# ============================================================
# CELL 10 — Explicit schema for student_enrollments.csv
# ============================================================

enrollment_schema = StructType([
    StructField("enrollment_id",            StringType(),  nullable=False),
    StructField("student_id",               StringType(),  nullable=True),
    StructField("course_id",                StringType(),  nullable=True),
    StructField("enrolled_date",            StringType(),  nullable=True),
    StructField("expected_completion_date", StringType(),  nullable=True),
    StructField("current_progress_pct",     DoubleType(),  nullable=True),
    StructField("last_accessed_date",       StringType(),  nullable=True),
    StructField("status",                   StringType(),  nullable=True),
])

print("✅ Enrollment schema defined with", len(enrollment_schema.fields), "fields")

In [0]:
# ============================================================
# CELL 11 — Read daily delta enrollments CSV + add metadata columns
# ============================================================

enroll_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .schema(enrollment_schema)
    .load(ENROLLMENTS_RAW_PATH)
)

# Add metadata columns before merge
enroll_staged = (
    enroll_raw
    .withColumn("_source_file",      F.input_file_name())
    .withColumn("_load_ts",          F.current_timestamp())
    .withColumn("_schema_version",   F.lit(SCHEMA_VERSION))
    .withColumn("_last_modified_ts", F.current_timestamp())
    # Cast date strings to proper DateType
    .withColumn("enrolled_date",            F.to_date(F.col("enrolled_date")))
    .withColumn("expected_completion_date", F.to_date(F.col("expected_completion_date")))
    .withColumn("last_accessed_date",       F.to_date(F.col("last_accessed_date")))
)

print(f"ℹ  Staged rows from source file: {enroll_staged.count():,}")

In [0]:
# ============================================================
# CELL 12 — MERGE INTO: upsert enrollments on enrollment_id
#
# Why MERGE and not APPEND?
#   - Same enrollment_id can appear multiple times across daily delta files
#     (student status changes: ACTIVE → PAUSED → DROPPED)
#   - MERGE ensures we always have the latest status without duplicates
#
# WHEN MATCHED   → update all columns (status, progress_pct, last_accessed)
# WHEN NOT MATCHED → insert as new enrollment
# ============================================================

# Check if target table exists — create it on first run, merge on subsequent runs
table_exists = spark.catalog.tableExists(ENROLL_BRONZE_TABLE)

if not table_exists:
    # First run: write directly, MERGE requires an existing target table
    (
        enroll_staged.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(ENROLL_BRONZE_TABLE)
    )
    print(f"✅ First run — created {ENROLL_BRONZE_TABLE} with {enroll_staged.count():,} rows")

else:
    # Subsequent runs: MERGE INTO using DeltaTable API
    target = DeltaTable.forName(spark, ENROLL_BRONZE_TABLE)

    (
        target.alias("tgt")
        .merge(
            enroll_staged.alias("src"),
            "tgt.enrollment_id = src.enrollment_id"   # join key
        )
        .whenMatchedUpdate(set={
            # Update all mutable fields when enrollment_id already exists
            "student_id":               "src.student_id",
            "course_id":                "src.course_id",
            "enrolled_date":            "src.enrolled_date",
            "expected_completion_date": "src.expected_completion_date",
            "current_progress_pct":     "src.current_progress_pct",
            "last_accessed_date":       "src.last_accessed_date",
            "status":                   "src.status",
            "_source_file":             "src._source_file",
            "_load_ts":                 "src._load_ts",
            "_last_modified_ts":        "src._last_modified_ts",
        })
        .whenNotMatchedInsertAll()    # insert all columns for new enrollment_ids
        .execute()
    )
    print(f"✅ MERGE INTO complete on {ENROLL_BRONZE_TABLE}")

print(f"   Total rows now: {spark.table(ENROLL_BRONZE_TABLE).count():,}")

In [0]:
# ============================================================
# CELL 13 — Quick verification of enrollments_bronze
# ============================================================

enroll_df = spark.table(ENROLL_BRONZE_TABLE)

print("── enrollments_bronze ────────────────────────────")
print(f"Total rows   : {enroll_df.count():,}")
print(f"Columns      : {len(enroll_df.columns)}")
print()

print("── status distribution ───────────────────────────")
enroll_df.groupBy("status").count().orderBy(F.desc("count")).show(truncate=False)

print("── progress_pct stats ────────────────────────────")
enroll_df.select(
    F.min("current_progress_pct").alias("min_pct"),
    F.max("current_progress_pct").alias("max_pct"),
    F.round(F.avg("current_progress_pct"), 2).alias("avg_pct")
).show()

print("── metadata sample ───────────────────────────────")
enroll_df.select(
    "enrollment_id", "status", "_load_ts", "_last_modified_ts"
).show(3, truncate=50)

## Part C — Data Quality Checks

Three DQ checks for this task:
1. `score_obtained` must be between 0 and `max_score`
2. `status = IN_PROGRESS` with non-null `submit_ts` is a logical inconsistency
3. `student_id` and `course_id` must not be NULL in either table
4. Referential integrity: all `course_id` values in enrollments must exist in `course_catalog_bronze`

All failures are written to `audit.dq_audit` — rows are NOT dropped from bronze.

In [0]:
# ============================================================
# CELL 15 — DQ Check 1: 0 <= score_obtained <= max_score
# Business rule from Task 1.2 spec
# ============================================================

invalid_scores = (
    spark.table(QUIZ_BRONZE_TABLE)
    .filter(
        (F.col("score_obtained") < 0) |
        (F.col("score_obtained") > F.col("max_score")) |
        F.col("score_obtained").isNull()
    )
    .withColumn("dq_check_name", F.lit("invalid_score_range"))
    .withColumn("dq_table",      F.lit(QUIZ_BRONZE_TABLE))
    .withColumn("dq_severity",   F.lit("HIGH"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("score_obtained="), F.col("score_obtained").cast("string"),
        F.lit("max_score="),      F.col("max_score").cast("string")
    ))
)

invalid_count = invalid_scores.count()
print("── DQ Check 1: score range validation ────────────")
print(f"   Invalid score rows: {invalid_count}")

if invalid_count > 0:
    (
        invalid_scores
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "attempt_id", "student_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {invalid_count} rows flagged → written to {DQ_AUDIT_TABLE}")
    invalid_scores.select(
        "attempt_id", "score_obtained", "max_score", "dq_message"
    ).show(5, truncate=False)
else:
    print("   ✅ PASSED — all scores within valid range")

In [0]:
# ============================================================
# CELL 16 — DQ Check 2: status=IN_PROGRESS + submit_ts IS NOT NULL
# This is a logical inconsistency — a submitted quiz cannot be IN_PROGRESS
# Flag these rows but do NOT remove them from bronze
# ============================================================

inconsistent = (
    spark.table(QUIZ_BRONZE_TABLE)
    .filter(
        (F.col("status") == "IN_PROGRESS") &
        (F.col("submit_ts").isNotNull())
    )
    .withColumn("dq_check_name", F.lit("in_progress_with_submit_ts"))
    .withColumn("dq_table",      F.lit(QUIZ_BRONZE_TABLE))
    .withColumn("dq_severity",   F.lit("MEDIUM"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("status=IN_PROGRESS but submit_ts is not null:"),
        F.col("submit_ts").cast("string"),
        F.lit("attempt_id="), F.col("attempt_id")
    ))
)

inconsistent_count = inconsistent.count()
print("── DQ Check 2: IN_PROGRESS + submit_ts inconsistency ──")
print(f"   Inconsistent rows: {inconsistent_count}")

if inconsistent_count > 0:
    (
        inconsistent
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "attempt_id", "student_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {inconsistent_count} rows flagged → written to {DQ_AUDIT_TABLE}")
    inconsistent.select(
        "attempt_id", "status", "submit_ts"
    ).show(5, truncate=False)
else:
    print("   ✅ PASSED — no IN_PROGRESS rows with a submit_ts found")

In [0]:
# ============================================================
# CELL 17 — DQ Check 3a: NULL student_id / course_id in both tables
#           DQ Check 3b: course_id referential integrity
#                        (all course_ids in enrollments must exist in catalog)
# ============================================================

# ── 3a: NULL checks ────────────────────────────────────────
def check_nulls(table_name, cols):
    df = spark.table(table_name)
    total = df.count()
    print(f"── NULL check: {table_name.split('.')[-1]} ─────────────")
    for col_name in cols:
        null_count = df.filter(F.col(col_name).isNull()).count()
        pct = (null_count / total * 100) if total > 0 else 0
        status = "✅" if null_count == 0 else "⚠"
        alert  = "  ← ALERT: exceeds 0.5% threshold!" if pct > 0.5 else ""
        print(f"   {status}  {col_name:<30} nulls={null_count:,}  ({pct:.3f}%){alert}")
    print()

check_nulls(QUIZ_BRONZE_TABLE,   ["student_id", "course_id"])
check_nulls(ENROLL_BRONZE_TABLE, ["student_id", "course_id"])

# ── 3b: Referential integrity — course_id in enrollments vs catalog ──
print("── Referential integrity: enrollments.course_id vs catalog ──")

# Only run if catalog table exists (it will after Task 1.3)
catalog_exists = spark.catalog.tableExists(CATALOG_BRONZE_TABLE)

if not catalog_exists:
    print("   ℹ  course_catalog_bronze not yet loaded (run after Task 1.3)")
    print("   ℹ  Skipping referential integrity check — will run in Task 1.3")
else:
    valid_course_ids = (
        spark.table(CATALOG_BRONZE_TABLE)
        .select("course_id")
        .distinct()
    )
    orphan_enrollments = (
        spark.table(ENROLL_BRONZE_TABLE)
        .join(valid_course_ids, on="course_id", how="left_anti")  # rows NOT in catalog
    )
    orphan_count = orphan_enrollments.count()

    if orphan_count > 0:
        print(f"   ⚠ {orphan_count} enrollment rows have course_id NOT in course_catalog")
        orphan_enrollments.select(
            "enrollment_id", "course_id", "student_id"
        ).show(5, truncate=False)

        # Write to audit
        (
            orphan_enrollments
            .withColumn("dq_check_name", F.lit("referential_integrity_course_id"))
            .withColumn("dq_table",      F.lit(ENROLL_BRONZE_TABLE))
            .withColumn("dq_severity",   F.lit("HIGH"))
            .withColumn("dq_checked_at", F.current_timestamp())
            .withColumn("dq_message",    F.concat_ws(" | ",
                F.lit("course_id not found in catalog:"), F.col("course_id")
            ))
            .select("dq_check_name", "dq_table", "dq_severity",
                    "dq_checked_at", "dq_message", "enrollment_id", "course_id")
            .write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(DQ_AUDIT_TABLE)
        )
    else:
        print("   ✅ PASSED — all enrollment course_ids exist in course_catalog_bronze")

In [0]:
# ============================================================
# CELL 18 — Task 1.2 completion summary
# ============================================================

quiz_count   = spark.table(QUIZ_BRONZE_TABLE).count()
enroll_count = spark.table(ENROLL_BRONZE_TABLE).count()

print("═" * 55)
print("  TASK 1.2 COMPLETE — Bronze Quiz + Enrollments")
print("═" * 55)
print()
print(f"  ✅ {QUIZ_BRONZE_TABLE}")
print(f"      Rows      : {quiz_count:,}")
print(f"      Method    : Incremental watermark on submit_ts")
print(f"      Partition : attempt_date")
print(f"      Metadata  : _source_file · _load_ts · _schema_version")
print()
print(f"  ✅ {ENROLL_BRONZE_TABLE}")
print(f"      Rows      : {enroll_count:,}")
print(f"      Method    : MERGE INTO on enrollment_id")
print(f"      Metadata  : _source_file · _load_ts · _last_modified_ts")
print()
print("  DQ checks run:")
print("      Cell 15 — score_obtained between 0 and max_score")
print("      Cell 16 — IN_PROGRESS status with non-null submit_ts")
print("      Cell 17 — NULL student_id / course_id + referential integrity")
print()
print("  ─────────────────────────────────────────────────")
print("  Next: Task 1.3 → 03_bronze_discussion_catalog")
print("         Discussion NDJSON + Course Catalog XLSX ingestion")
print("═" * 55)